## Model Building

In [7]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", message=".*no associated frequency.*")

df2 = pd.read_csv("../datasets/eda_cleaned.csv", parse_dates=["date"])
df2["total"] = pd.to_numeric(df2["total"], errors="coerce")
df2 = df2.dropna(subset=["date", "total"]).sort_values("date").set_index("date")

print(f"Loaded {len(df2)} rows from ../datasets/eda_cleaned.csv")
print(f"Date range: {df2.index.min().date()} to {df2.index.max().date()}")


Loaded 999 rows from ../datasets/eda_cleaned.csv
Date range: 2022-11-01 to 2026-02-28


In [8]:
df3 = df2.copy()
df3.index = pd.to_datetime(df2.index)  
df3 = df3.sort_index()

In [3]:
df3.head()

,day,time,total,total_employees_hours,total_employees,total_tips,month,week,year,quarter,...,total:30_days_rolling,total:50_days_rolling,is_future,is_closed,is_suspicious_zero,store_open,weekly_total_diff,day_of_week,month_name,week_of_year
date,,,,,,,,,,,,,,,,,,,,,
2022-11-01,Tuesday,11 to 8,147.0,8.5,0,0.0,11,44,2022,4,...,147.00,147.00,False,False,False,1,NaN,Tuesday,November,44
2022-11-02,Wednesday,11 to 8,81.0,0.0,0,0.0,11,44,2022,4,...,114.00,114.00,False,False,False,1,NaN,Wednesday,November,44
2022-11-03,Thursday,11 to 8,18.0,0.0,0,0.0,11,44,2022,4,...,82.00,82.00,False,False,False,1,NaN,Thursday,November,44
2022-11-04,Friday,11 to 9,27.0,9.0,0,0.0,11,44,2022,4,...,68.25,68.25,False,False,False,1,NaN,Friday,November,44
2022-11-05,Saturday,10 to 9,260.0,10.0,0,0.0,11,44,2022,4,...,106.60,106.60,False,False,False,1,NaN,Saturday,November,44


In [4]:
df3.index.min()

Timestamp('2022-11-01 00:00:00')

In [5]:
df3.index.max()

Timestamp('2026-01-07 00:00:00')

In [6]:
df3[["total", "total:10_days_rolling"]].head(15)

,total,total:10_days_rolling
date,,
2022-11-01,147.0,147.000000
2022-11-02,81.0,114.000000
2022-11-03,18.0,82.000000
2022-11-04,27.0,68.250000
2022-11-05,260.0,106.600000
2022-11-06,81.0,102.333333
2022-11-07,306.0,131.428571
2022-11-08,123.0,130.375000
2022-11-09,90.0,125.888889


In [ ]:
# Drop unsafe or redundant columns
cols_to_drop = [
    "is_future",
    "total:10_days_rolling",
    "total:30_days_rolling",
    "total:50_days_rolling",
    "weekly_total_diff"
    "day",          # redundant (we have date index)
    "time",         # constant? not useful for daily
    "month_name"   # redundant with month
]

df3 = df3.drop(columns=[c for c in cols_to_drop if c in df3.columns])


In [ ]:
df3.columns

In [ ]:
# Lags
df3["lag_1"] = df3["total"].shift(1)
df3["lag_7"] = df3["total"].shift(7)
df3["lag_14"] = df3["total"].shift(14)

# Rolling means (shift first to avoid leakage)
df3["rolling_7"] = df3["total"].shift(1).rolling(7).mean()
df3["rolling_30"] = df3["total"].shift(1).rolling(30).mean()

# Rolling std (captures volatility)
df3["rolling_7_std"] = df3["total"].shift(1).rolling(7).std()


In [ ]:
df3 = df3.dropna()


In [ ]:
cols_to_drop = [
    "day",
    "day_of_week"        # redundant (we have date index
]
df3 = df3.drop(columns=cols_to_drop)

In [ ]:

df3.info()

In [ ]:
test_size = 60
val_size = 60

train = df3.iloc[:-(test_size + val_size)]
val   = df3.iloc[-(test_size + val_size):-test_size]
test  = df3.iloc[-test_size:]


In [ ]:
# Baseline 1: Naive

from sklearn.metrics import mean_absolute_error

naive_pred = test["lag_1"]
mae_naive = mean_absolute_error(test["total"], naive_pred)

print("Naive MAE:", mae_naive)

In [ ]:
# Seasonal Naive

seasonal_pred = test["lag_7"]
mae_seasonal = mean_absolute_error(test["total"], seasonal_pred)

print("Seasonal MAE:", mae_seasonal)


In [ ]:
df3.columns

In [ ]:
target = "total"

features = [col for col in df3.columns if col != target]

X_train = train[features]
y_train = train[target]

X_val = val[features]
y_val = val[target]

X_test = test[features]
y_test = test[target]


In [ ]:
df3.head()

In [ ]:
# Ridge Regression

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

model.fit(X_train, y_train)

val_pred = model.predict(X_val)
test_pred = model.predict(X_test)

mae_val = mean_absolute_error(y_val, val_pred)
mae_test = mean_absolute_error(y_test, test_pred)

print("Validation MAE:", mae_val)
print("Test MAE:", mae_test)


In [ ]:
residuals = y_test - test_pred

plt.figure(figsize=(10,4))
plt.plot(residuals)
plt.title("Residuals Over Time")
plt.show()

In [ ]:
print(df3[["total", "rolling_7"]].head(15))


In [ ]:
df3.corr()["total"].sort_values(ascending=False)

In [ ]:
df3["rolling_7"] = df3["total"].shift(1).rolling(7).mean()
df3["rolling_30"] = df3["total"].shift(1).rolling(30).mean()

In [ ]:
df3["rolling_7"]

In [ ]:
df3[["total", "weekly_total_diff", "lag_7"]].head(15)

In [ ]:
# Since there is a data leakage happening because of weekly_total_diff we are dropping the column along with
#  "total_employees_hours", total_employees, total_tips

df3 = df3.drop(columns = ["weekly_total_diff","total_employees_hours","total_employees","total_tips","is_suspicious_zero"])



In [ ]:
df3.columns

In [ ]:
df3.corr()["total"].sort_values(ascending=False)


In [ ]:
target = "total"

features = [col for col in df3.columns if col != target]

X_train = train[features]
y_train = train[target]

X_val = val[features]
y_val = val[target]

X_test = test[features]
y_test = test[target]


In [ ]:
test_size = 60
val_size = 60

train = df3.iloc[:-(test_size + val_size)]
val   = df3.iloc[-(test_size + val_size):-test_size]
test  = df3.iloc[-test_size:]


In [ ]:
mae_naive = mean_absolute_error(test["total"], test["lag_1"])
mae_seasonal = mean_absolute_error(test["total"], test["lag_7"])

print("Naive MAE:", mae_naive)
print("Seasonal MAE:", mae_seasonal)

In [ ]:
# Ridge Regression

model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

model.fit(X_train, y_train)

val_pred = model.predict(X_val)
test_pred = model.predict(X_test)

mae_val = mean_absolute_error(y_val, val_pred)
mae_test = mean_absolute_error(y_test, test_pred)

print("Validation MAE:", mae_val)
print("Test MAE:", mae_test)

In [ ]:
print(X_test.head())



In [ ]:
print(y_test.head())

### Checking seasonality within Data

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df3["total"])

In [ ]:
df3.groupby("week_of_year")["total"].mean().plot(figsize=(10,4))


In [ ]:
residuals = y_test - test_pred

plt.figure(figsize=(12,6))
plt.plot(residuals, label='Residuals')
# plt.plot(y_test, label='Actual', color='red')
# plt.plot(test_pred, label='Predicted', color='green')
plt.title("Residuals vs Actual vs Predicted")
plt.legend()
plt.show()

In [ ]:
df3["total"].autocorr(lag=7)

In [ ]:
df3["total"].rolling(30).mean().plot(figsize=(10,4))
plt.title("30-Day Rolling Mean")
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(df3["total"])
print("ADF Statistic:", result[0])
print("p-value:", result[1])

In [ ]:
# Cyclical Encoding for Seasonality
# Cyclical encoding → captures yearly wave
df3["month_sin"] = np.sin(2 * np.pi * df3["month"] / 12)
df3["month_cos"] = np.cos(2 * np.pi * df3["month"] / 12)

df3["week_sin"] = np.sin(2 * np.pi * df3["week_of_year"] / 52)
df3["week_cos"] = np.cos(2 * np.pi * df3["week_of_year"] / 52)

In [ ]:
df3["lag_21"] = df3["total"].shift(21)
df3["expanding_mean"] = df3["total"].shift(1).expanding().mean()

# Lag_21 → captures longer seasonal memory


# Expanding mean → captures long-term baseline shift

In [ ]:
%pip install lightgbm

In [ ]:
from lightgbm import LGBMRegressor

model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

model.fit(X_train, y_train)

val_pred = model.predict(X_val)
test_pred = model.predict(X_test)

print("Validation MAE:", mean_absolute_error(y_val, val_pred))
print("Test MAE:", mean_absolute_error(y_test, test_pred))

In [ ]:
df3.columns

In [ ]:
df3.groupby(df3.index.weekday)[["store_open", "is_closed"]].mean()

In [ ]:
%pip install holidays

In [ ]:
import holidays

us_holidays = holidays.US()

df3["is_holiday"] = df3.index.to_series().apply(lambda x: x in us_holidays)

In [ ]:
# Step-by-Step Recursive 7-Day Forecast

forecast_df = df3.copy()

n_days = 7
future_predictions = []

feature_cols = model.feature_name_

for i in range(n_days):
    
    last_date = forecast_df.index[-1]
    next_date = last_date + pd.Timedelta(days=1)
    
    new_row = pd.DataFrame(index=[next_date])
    
    # Calendar features (ONLY those used in training)
    new_row["month"] = next_date.month
    new_row["week"] = next_date.isocalendar().week
    new_row["year"] = next_date.year
    new_row["quarter"] = (next_date.month - 1)//3 + 1
    new_row["week_of_year"] = next_date.isocalendar().week
    new_row["is_weekend"] = int(next_date.weekday() >= 5)
    
    # Store schedule (based on your data behavior)
    new_row["store_open"] = 1
    new_row["is_closed"] = False
    
    # Lags (based on last known + predicted values)
    new_row["lag_1"] = forecast_df["total"].iloc[-1]
    new_row["lag_7"] = forecast_df["total"].iloc[-7]
    new_row["lag_14"] = forecast_df["total"].iloc[-14]
    
    # Rolling features
    new_row["rolling_7"] = forecast_df["total"].iloc[-7:].mean()
    new_row["rolling_30"] = forecast_df["total"].iloc[-30:].mean()
    new_row["rolling_7_std"] = forecast_df["total"].iloc[-7:].std()
    
    # Match exact feature order
    X_new = new_row[feature_cols]
    
    # Predict
    pred = model.predict(X_new)[0]
    
    new_row["total"] = pred
    
    forecast_df = pd.concat([forecast_df, new_row])
    future_predictions.append(pred)

# Build forecast series
future_dates = pd.date_range(
    start=df3.index[-1] + pd.Timedelta(days=1),
    periods=7
)

forecast_series = pd.Series(future_predictions, index=future_dates)

print(forecast_series)

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df3["total"].tail(60), label="Historical")
plt.plot(forecast_series, marker="o", label="7-Day Forecast")
plt.legend()
plt.show()

In [ ]:
print(forecast_series)

## Model 1: Linear Regession

In [ ]:
# features= ['total_employees','month', 'week', 'year', 'quarter']
features= ['total_employees','month', 'year', 'quarter']

In [ ]:
# Ensure chronological order (time-series split assumes this)
df2 = df2.sort_values('date').reset_index(drop=True)

# Avoid target leakage from imputation: keep only rows with known target
# (safe even if you already dropped missing totals earlier)
df2 = df2.dropna(subset=['total']).copy()

X = df2[features].copy()
y = df2['total'].copy()

In [ ]:
X.info()

In [ ]:
X.head()

In [ ]:
split_index = int(len(df2) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [ ]:
# (Removed) Casting to int here was ineffective because it happened AFTER the split.
# Keep features numeric; preprocessing is handled in the model pipeline below.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Linear Regression works better with scaling. Also impute any missing feature values.
lr_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

lr_model.fit(X_train, y_train)

In [ ]:
# Predictions
y_pred = lr_model.predict(X_test)

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df2.index[split_index:], y_test, label="Actual", marker='o')
plt.plot(df2.index[split_index:], y_pred, label="Predicted", marker='x')
plt.title("Linear Regression: Actual vs Predicted Sales")
plt.xlabel("Date")
plt.ylabel("Total Sales")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
rmse= round(np.sqrt(mse),2)
print('RMSE',rmse)

## Model 2: Random Forest

In [ ]:
df2.columns

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Train model
rf = RandomForestRegressor(
    n_estimators=200,  # number of trees
    max_depth=None,    # let trees expand fully
    random_state=42,
    n_jobs=-1          # use all CPU cores
)
rf.fit(X_train, y_train)

# Predictions
y_pred_rf = rf.predict(X_test)

In [ ]:
mse = mean_squared_error(y_test, y_pred_rf)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred_rf)
r2 = r2_score(y_test, y_pred_rf)
rmse= round(np.sqrt(mse),2)
print('RMSE',rmse)

print("Random Forest MSE:", mse)
print("Random Forest RMSE:", rmse)
print("Random Forest MAE:", mae)
print("Random Forest R²:", r2)


In [ ]:
feat_importances = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(feat_importances)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(df2['date'].iloc[split_index:], y_test, label="Actual", marker='o')
plt.plot(df2['date'].iloc[split_index:], y_pred_rf, label="Random Forest", marker='x')
plt.title("Random Forest: Actual vs Predicted Sales")
plt.xlabel("Date")
plt.ylabel("Total Sales")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## Model 3: ARIMA( Auto Regressuve Integrated Moving Average)

In [ ]:
# Ensure date is a clean datetime index (ARIMA can't handle NaT at the ends)
df = df2.copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Drop rows with missing dates (they become NaT and break pd.date_range / predict ranges)
df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

# For time-series modeling, also drop missing target values
sales = df["total"].dropna()

# Use date as index
sales = sales.copy()
sales.index = df.loc[sales.index, "date"]
sales = sales.sort_index()

In [ ]:
# ARIMA needs a stationary series (constant mean/variance). Use ADF test:

from statsmodels.tsa.stattools import adfuller

result = adfuller(sales.dropna())
print("ADF Statistic:", result[0])
print("p-value:", result[1])

In [ ]:
# Optional (not required): pmdarima auto_arima helper
# If you have it installed locally you can uncomment these lines.
# !pip install pmdarima

In [ ]:
# Optional: auto_arima (requires pmdarima)
try:
    from pmdarima import auto_arima
except ImportError:
    auto_arima = None
    print("pmdarima not installed; skipping auto_arima. (This is OK.)")

if auto_arima is not None:
    model = auto_arima(sales, seasonal=False, stepwise=True, trace=True)
    print(model.summary())

order = model.order
seasonal_order = model.seasonal_order

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# Example: If auto_arima suggested (2,1,2)
model = ARIMA(sales, order=order)
model_fit = model.fit()

print(model_fit.summary())

In [ ]:
# Use robust start/end dates (avoid NaT issues)
start_date = sales.index.min()
end_date = sales.index.max()

fitted_values = model_fit.predict(start=start_date, end=end_date)
forecast = model_fit.forecast(steps=30)

future_dates = pd.date_range(
    start=end_date + pd.Timedelta(days=1),
    periods=30,
    freq="D",
)

plt.figure(figsize=(20,10))
plt.plot(sales, label="Actual")
plt.plot(fitted_values, label="Fitted", color="orange")
plt.plot(future_dates, forecast, label="Forecast", color="red")
plt.legend()
plt.show()


mae = mean_absolute_error(sales, fitted_values)
rmse = np.sqrt(mean_squared_error(sales, fitted_values))

print(f"Sales MAE: {mae:.2f}")
print(f"Sales RMSE: {rmse:.2f}")

In [ ]:
print(sales.mean())

# SARIMAX MODEL

##### Since ARIMA predicted the daily sales properly but it couldnt pickup the spikes, holidays and the sale drops in between we will use SARIMAX

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
# Fit SARIMAX model
sarimax_model = SARIMAX(sales, 
                        order=order, 
                        seasonal_order=seasonal_order, 
                        enforce_stationarity=False, 
                        enforce_invertibility=False)

sarimax_result = sarimax_model.fit(disp=False)

print(sarimax_result.summary())

In [ ]:

# Use robust start/end dates (avoid NaT issues)
start_date = sales.index.min()
end_date = sales.index.max()

fitted_values = sarimax_result.predict(start=start_date, end=end_date)
forecast = sarimax_result.forecast(steps=30)

future_dates = pd.date_range(
    start=end_date + pd.Timedelta(days=1),
    periods=30,
    freq="D",
)

plt.figure(figsize=(20,10))
plt.plot(sales, label="Actual")
plt.plot(fitted_values, label="Fitted", color="orange")
plt.plot(future_dates, forecast, label="Forecast", color="red")
plt.legend()
plt.show()


mae = mean_absolute_error(sales, fitted_values)
rmse = np.sqrt(mean_squared_error(sales, fitted_values))

print(f"Sales MAE: {mae:.2f}")
print(f"Sales RMSE: {rmse:.2f}")

## Model 4: AR Model

In [ ]:
df2.info()

In [ ]:
# # Ensure date is index
df = df2.sort_values("date")
df.set_index("date", inplace=True)

# Select only sales column
sales = df["total"]

In [ ]:
train_data, test_data = sales[:-100], sales[-100:]

In [ ]:
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_squared_error

# Ensure date is index
df = df2.sort_values("date")
df.set_index("date", inplace=True)

# Select only the sales column
sales = df["total"]

# Split into train and test
train_data, test_data = sales[:-100], sales[-100:]

# Fit AutoReg model
model = AutoReg(train_data, lags=30)
model_fit = model.fit()

# Make predictions with fitted model
predictions = model_fit.predict(
    start=len(train_data),
    end=len(train_data) + len(test_data) - 1,
    dynamic=False
)

# Plot actual vs predicted
plt.figure(figsize=(10,5))
plt.plot(test_data.index, test_data, label='Test Data')
plt.plot(test_data.index, predictions, color='red', linestyle='--', label='Predicted Values')
plt.title('Total Sales: Actual vs Predicted')
plt.xlabel('Date')
plt.ylabel('Total Sales')
plt.legend()
plt.show()

# RMSE
rmse = round(np.sqrt(mean_squared_error(test_data, predictions)), 2)
print('RMSE:', rmse)


# Model 5: MA Model

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error

# Ensure date is index
df = df2.sort_values("date")
df.set_index("date", inplace=True)

# Select only the sales column
sales = df["total"]

# Split into train and test
train_data, test_data = sales[:-100], sales[-100:]

# Fit MA model for that in order make the first 2 parameter that is AR and I and change the third value ie MA
model = ARIMA(train_data, order=(0,0,30))
model_fit = model.fit()

# Make predictions with fitted model
predictions = model_fit.predict(
    start=len(train_data),
    end=len(train_data) + len(test_data) - 1,
    dynamic=False
)

# Plot actual vs predicted
plt.figure(figsize=(10,5))
plt.plot(test_data.index, test_data, label='Test Data')
plt.plot(test_data.index, predictions, color='red', linestyle='--', label='Predicted Values')
plt.title('Total Sales: Actual vs Predicted')
plt.xlabel('Date')
plt.ylabel('Total Sales')
plt.legend()
plt.show()

# RMSE
rmse = round(np.sqrt(mean_squared_error(test_data, predictions)), 2)
print('RMSE:', rmse)


# Model 6: ARMA MODEL

In [ ]:
%%time
# Fit ARMA model for that in order make the 2nd parameter that is I and change the first and third value ie ARMA
model = ARIMA(train_data, order=(10,0,30))
model_fit = model.fit()

# Make predictions with fitted model
predictions = model_fit.predict(
    start=len(train_data),
    end=len(train_data) + len(test_data) - 1,
    dynamic=False
)

# Plot actual vs predicted
plt.figure(figsize=(10,5))
plt.plot(test_data.index, test_data, label='Test Data')
plt.plot(test_data.index, predictions, color='red', linestyle='--', label='Predicted Values')
plt.title('Total Sales: Actual vs Predicted')
plt.xlabel('Date')
plt.ylabel('Total Sales')
plt.legend()
plt.show()

# RMSE
rmse = round(np.sqrt(mean_squared_error(test_data, predictions)), 2)
print('RMSE:', rmse)


In [ ]:
from statsmodels.tsa.stattools import arma_order_select_ic
from statsmodels.tools.sm_exceptions import ConvergenceWarning
import warnings

# Suppress convergence warnings
warnings.simplefilter("ignore", ConvergenceWarning)

sel = arma_order_select_ic(train_data, max_ar=5, max_ma=5, ic=['aic','bic'])
print("Best order by AIC:", sel.aic_min_order)
print("Best order by BIC:", sel.bic_min_order)


### The RMSE is way to high for a sales data within ranging 300-1500 is bad

In [ ]:
# Fit ARMA model for that in order make the 2nd parameter that is I and change the first and third value ie ARMA
model = ARIMA(train_data, order=(10,1,30))
model_fit = model.fit()

# Make predictions with fitted model
predictions = model_fit.predict(
    start=len(train_data),
    end=len(train_data) + len(test_data) - 1,
    dynamic=False
)

# Plot actual vs predicted
plt.figure(figsize=(10,5))
plt.plot(test_data.index, test_data, label='Test Data')
plt.plot(test_data.index, predictions, color='red', linestyle='--', label='Predicted Values')
plt.title('Total Sales: Actual vs Predicted')
plt.xlabel('Date')
plt.ylabel('Total Sales')
plt.legend()
plt.show()

# RMSE
rmse = round(np.sqrt(mean_squared_error(test_data, predictions)), 2)
print('RMSE:', rmse)


## Best Model: ML Forecasting (lags + rolling + calendar)

This section builds a proper daily time series, compares against strong baselines, and trains a boosted model with time-series validation.


In [ ]:
import numpy as np
import pandas as pd

from pandas.tseries.holiday import USFederalHolidayCalendar

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import HistGradientBoostingRegressor


def smape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return 100.0 * np.mean(2.0 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + eps))


def regression_report(name, y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    s = float(smape(y_true, y_pred))
    acc = float(100.0 - s)  # "accuracy" as (100 - sMAPE)
    print(f"{name}\n  RMSE: {rmse:.2f}\n  MAE:  {mae:.2f}\n  sMAPE: {s:.2f}%\n  Forecast Accuracy (100-sMAPE): {acc:.2f}%\n")


# 1) Load and create DAILY total series
raw = pd.read_excel("../merged_data.xlsx")
raw["date"] = pd.to_datetime(raw.get("date"), errors="coerce")
raw["total"] = pd.to_numeric(raw.get("total"), errors="coerce")
raw = raw.dropna(subset=["date"]).copy()

# Aggregate to 1 row per day (handles duplicate dates)
daily = raw.groupby("date", as_index=True)["total"].sum().sort_index()

# Build continuous daily index (needed for true day-based lags)
full_idx = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
y_raw = daily.reindex(full_idx)              # NaN where missing day

y_filled = y_raw.fillna(0.0)                # assume missing day = 0 sales (closed/missing)


# 2) Feature engineering (calendar + lags + rolling stats)
feat = pd.DataFrame(index=full_idx)
feat["y"] = y_raw
feat["y_filled"] = y_filled

feat["dow"] = feat.index.dayofweek
feat["month"] = feat.index.month
feat["week"] = feat.index.isocalendar().week.astype(int)
feat["year"] = feat.index.year
feat["is_weekend"] = (feat["dow"] >= 5).astype(int)

holidays = USFederalHolidayCalendar().holidays(start=feat.index.min(), end=feat.index.max())
feat["is_holiday"] = feat.index.normalize().isin(holidays).astype(int)

# Lags
for lag in [1, 7, 14, 28]:
    feat[f"lag_{lag}"] = feat["y_filled"].shift(lag)

# Rolling features (shifted to avoid leakage)
feat["roll_mean_7"] = feat["y_filled"].shift(1).rolling(7).mean()
feat["roll_mean_28"] = feat["y_filled"].shift(1).rolling(28).mean()
feat["roll_std_7"] = feat["y_filled"].shift(1).rolling(7).std()
feat["weekly_change"] = feat["y_filled"].shift(1) - feat["y_filled"].shift(8)

# Keep ONLY days where the target is known
feat = feat[feat["y"].notna()].copy()

# Drop early rows where lag/rolling features are missing
required = ["lag_28", "roll_mean_28"]
feat = feat.dropna(subset=required)

X = feat.drop(columns=["y"])
y = feat["y"]


# 3) Holdout evaluation (last 90 days)
TEST_DAYS = 90
split = max(1, len(feat) - TEST_DAYS)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# Strong baselines
pred_naive_1 = X_test["lag_1"].values
pred_naive_7 = X_test["lag_7"].values

regression_report("Baseline: Naive (t-1)", y_test, pred_naive_1)
regression_report("Baseline: Seasonal Naive (t-7)", y_test, pred_naive_7)


# 4) Boosted model (usually strong for sales)
model = HistGradientBoostingRegressor(
    random_state=42,
    max_depth=6,
    learning_rate=0.05,
    max_iter=600,
)
model.fit(X_train, y_train)
pred = model.predict(X_test)

regression_report("Model: HistGradientBoosting (lags+rolling+calendar)", y_test, pred)


# 5) Walk-forward validation on train (time-series CV)
tscv = TimeSeriesSplit(n_splits=5)
cv_rmse = []
cv_smape = []

for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_train), 1):
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

    m = HistGradientBoostingRegressor(
        random_state=42,
        max_depth=6,
        learning_rate=0.05,
        max_iter=600,
    )
    m.fit(X_tr, y_tr)
    p = m.predict(X_va)

    cv_rmse.append(np.sqrt(mean_squared_error(y_va, p)))
    cv_smape.append(smape(y_va, p))

print("TimeSeriesSplit (train only)")
print(f"  Avg RMSE:  {np.mean(cv_rmse):.2f} ± {np.std(cv_rmse):.2f}")
print(f"  Avg sMAPE: {np.mean(cv_smape):.2f}% ± {np.std(cv_smape):.2f}%")


### Optional: Quick tuning + log-transform (often improves RMSE)

Runs a tiny search over a few settings and picks the best by time-series CV.


In [ ]:
from sklearn.compose import TransformedTargetRegressor

param_grid = [
    {"max_depth": 3, "learning_rate": 0.05, "max_iter": 400},
    {"max_depth": 4, "learning_rate": 0.05, "max_iter": 600},
    {"max_depth": 6, "learning_rate": 0.05, "max_iter": 600},
    {"max_depth": 6, "learning_rate": 0.03, "max_iter": 900},
]

candidates = []

# Two variants: plain + log1p(target)
for params in param_grid:
    candidates.append((
        f"HistGBR plain {params}",
        HistGradientBoostingRegressor(random_state=42, **params),
    ))
    candidates.append((
        f"HistGBR log1p {params}",
        TransformedTargetRegressor(
            regressor=HistGradientBoostingRegressor(random_state=42, **params),
            func=np.log1p,
            inverse_func=np.expm1,
        ),
    ))

best = None

tscv = TimeSeriesSplit(n_splits=5)

for name, cand in candidates:
    rmses = []
    for tr_idx, va_idx in tscv.split(X_train):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        cand.fit(X_tr, y_tr)
        p = cand.predict(X_va)
        rmses.append(np.sqrt(mean_squared_error(y_va, p)))

    avg = float(np.mean(rmses))
    std = float(np.std(rmses))
    print(f"{name}\n  CV RMSE: {avg:.2f} ± {std:.2f}\n")

    if best is None or avg < best[0]:
        best = (avg, std, name, cand)

best_rmse, best_std, best_name, best_model = best
print("=" * 60)
print(f"BEST by CV RMSE: {best_name}")
print(f"CV RMSE: {best_rmse:.2f} ± {best_std:.2f}")

# Fit best model on full train and evaluate holdout
best_model.fit(X_train, y_train)
best_pred = best_model.predict(X_test)
regression_report(f"Best model holdout ({best_name})", y_test, best_pred)
